# Student 360 in 1 Hour - Cleanup
⏱ **~2 min.** Run this only when you are finished with the lab.

⚠️ Removes **only** the `STUDENT360_LAB` objects: your agents from the shared CoWork object, your persona role grants, the budget, the database, warehouse and roles. The shared CoWork object and other teams' agents are left untouched.

In [ ]:
%%sql -r set_context
USE ROLE ACCOUNTADMIN;
SELECT CURRENT_ROLE();

## 1. Unpublish the lab agents from CoWork
🛠️ Only your own agents are dropped; the shared object is never replaced or removed.

In [ ]:
%%sql -r unpublish
USE ROLE ACCOUNTADMIN;
-- Each step tolerates objects that were never created, so cleanup works even if you stopped early.
EXECUTE IMMEDIATE $$
BEGIN
  BEGIN ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT DROP AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT; EXCEPTION WHEN OTHER THEN NULL; END;
  BEGIN ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT DROP AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST; EXCEPTION WHEN OTHER THEN NULL; END;
  RETURN 'Lab agents removed from CoWork (if they were published)';
END;
$$;

## 2. Remove persona roles from your user

In [ ]:
%%sql -r revoke_me
USE ROLE ACCOUNTADMIN;
EXECUTE IMMEDIATE $$
DECLARE me STRING DEFAULT CURRENT_USER();
BEGIN
  BEGIN EXECUTE IMMEDIATE 'REVOKE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING FROM USER "' || me || '"'; EXCEPTION WHEN OTHER THEN NULL; END;
  BEGIN EXECUTE IMMEDIATE 'REVOKE ROLE STUDENT360_LAB_IR_ANALYST FROM USER "' || me || '"'; EXCEPTION WHEN OTHER THEN NULL; END;
  RETURN 'Persona roles revoked from ' || me || ' (if granted)';
END;
$$;

## 3. Drop the budget

In [ ]:
%%sql -r drop_cost_objects
USE ROLE ACCOUNTADMIN;
EXECUTE IMMEDIATE $$
BEGIN
  BEGIN DROP SNOWFLAKE.CORE.BUDGET IF EXISTS STUDENT360_LAB.AGENTS.AI_BUDGET; EXCEPTION WHEN OTHER THEN NULL; END;
  RETURN 'Budget dropped (if it existed)';
END;
$$;

## 4. Drop the database, warehouse and roles
🔹 Dropping the database removes agents, semantic view, search services, dynamic tables, policies and tags in one step.

In [ ]:
%%sql -r drop_database
USE ROLE ACCOUNTADMIN;
DROP DATABASE IF EXISTS STUDENT360_LAB;
DROP WAREHOUSE IF EXISTS STUDENT360_LAB_WH;

In [ ]:
%%sql -r drop_roles
USE ROLE ACCOUNTADMIN;
DROP ROLE IF EXISTS STUDENT360_LAB_ADVISOR_ENGINEERING;
DROP ROLE IF EXISTS STUDENT360_LAB_IR_ANALYST;
DROP ROLE IF EXISTS STUDENT360_LAB_ADMIN;

## 5. Verify
📌 The first three commands should return no rows. The last may still list other teams' agents in the shared CoWork object; that is expected.

In [ ]:
%%sql -r verify_clean
USE ROLE ACCOUNTADMIN;
SHOW DATABASES LIKE 'STUDENT360_LAB';
SHOW WAREHOUSES LIKE 'STUDENT360_LAB%';
SHOW ROLES LIKE 'STUDENT360_LAB%';
SHOW AGENTS IN SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT;

## ✅ Clean
Nothing from the lab remains, and the shared CoWork object is as you found it.